In [1]:
print('hiii')

hiii


In [2]:
! uv pip install langchain langchain-community langchain-google-genai langchain-huggingface sentence-transformers faiss-cpu python-dotenv rapidocr-onnxruntime

Using Python 3.11.9 environment at: C:\Users\Safnas\OneDrive\Desktop\Proj\llmops-pipeline\LLMOps-pipeline\.venv
Resolved 107 packages in 7.34s
 Downloaded setuptools
 Downloaded shapely
 Downloaded networkx
 Downloaded pydantic-core
 Downloaded sqlalchemy
 Downloaded langchain-community
 Downloaded tokenizers
 Downloaded cryptography
 Downloaded hf-xet
 Downloaded sympy
 Downloaded pillow
 Downloaded scikit-learn
 Downloaded transformers
 Downloaded numpy
 Downloaded onnxruntime
 Downloaded rapidocr-onnxruntime
 Downloaded faiss-cpu
 Downloaded scipy
 Downloaded opencv-python
 Downloaded torch
Prepared 101 packages in 3m 02s
Installed 101 packages in 16.08s
 + aiohappyeyeballs==2.6.1
 + aiohttp==3.13.5
 + aiosignal==1.4.0
 + annotated-doc==0.0.4
 + annotated-types==0.7.0
 + anyio==4.13.0
 + attrs==26.1.0
 + certifi==2026.4.22
 + cffi==2.0.0
 + charset-normalizer==3.4.7
 + click==8.4.0
 + cryptography==48.0.0
 + dataclasses-json==0.6.7
 + distro==1.9.0
 + faiss-cpu==1.13.2
 + filelock==

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

# LangChain's Gemini integration checks GOOGLE_API_KEY first.
# If your .env uses GEMINI_API_KEY, this maps it automatically.
google_api_key = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")

if not google_api_key:
    raise ValueError("Add GOOGLE_API_KEY=your_gemini_api_key or GEMINI_API_KEY=your_gemini_api_key to your .env file")

os.environ["GOOGLE_API_KEY"] = google_api_key

## Data Ingestion


In [5]:
from langchain_community.document_loaders import PyPDFLoader

c:\Users\Safnas\OneDrive\Desktop\Proj\llmops-pipeline\LLMOps-pipeline\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
loader = PyPDFLoader("C:/Users/Safnas/OneDrive/Desktop/Proj/llmops-pipeline/LLMOps-pipeline/data/BCWEEK2.pdf")
documents = loader.load()

In [16]:
documents[0].page_content[:500]  # Print the first 500 characters of the first documen

'Business \nCommunication\n(HS-218)\n Today’s topic: BC Foundations (Contd.)\nSE Computer Science – Fall 2022\nThursday, October 20, 2022\nWeek 2\nLecture 4-6\nLecturer Zermeena Khan'

In [21]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [22]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=200,chunk_overlap=20)

In [23]:
text_chunks=text_splitter.split_documents(documents)

In [24]:
text_chunks

[Document(metadata={'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'source': 'C:/Users/Safnas/OneDrive/Desktop/Proj/llmops-pipeline/LLMOps-pipeline/data/BCWEEK2.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Business \nCommunication\n(HS-218)\n Today’s topic: BC Foundations (Contd.)\nSE Computer Science – Fall 2022\nThursday, October 20, 2022\nWeek 2\nLecture 4-6\nLecturer Zermeena Khan'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'source': 'C:/Users/Safnas/OneDrive/Desktop/Proj/llmops-pipeline/LLMOps-pipeline/data/BCWEEK2.pdf', 'total_pages': 15, 'page': 1, 'page_label': '2'}, page_content='What we will cover?\n BC (HS-218)\nWhat are teams?\nMeetings\nListening\nNon-Verbal Communication\nSE Computer Science – Fall 2022'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'source': 'C:/Users/Safnas/OneDrive/Desktop/Proj/llmops-pipeline/LLMOps-pipeline/data/BCWEEK2.pdf', 'total_

In [25]:
! uv pip install faiss-cpu

Using Python 3.11.9 environment at: C:\Users\Safnas\OneDrive\Desktop\Proj\llmops-pipeline\LLMOps-pipeline\.venv
Resolved 3 packages in 537ms
Installed 2 packages in 1.07s
 + faiss-cpu==1.13.2
 + numpy==2.4.5


In [29]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

In [30]:
# Free local embedding model. No OpenAI key needed.
# Dimension: 384
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5063.15it/s]


In [31]:
vectorstore=FAISS.from_documents(text_chunks, embeddings)

In [32]:
vectorstore

In [33]:
retriever=vectorstore.as_retriever()

In [34]:
# Perform similarity search
query = "What is the Key Characteristics of LISTENING process?"
docs = vectorstore.similarity_search(query, k=4)

# Display the results
for i, doc in enumerate(docs):
    print(f"Document {i+1}:")
    print(doc.page_content)
    print("-" * 50)


Document 1:
LISTENING MODES
 BC (HS-218)
SE Computer Science – Fall 2022
•Focusing on information purely
•No evaluation, neutralContent Listening
•Understanding the logic
•Strength of the evidence
--------------------------------------------------
Document 2:
The LISTENING process
 BC (HS-218)
SE Computer Science – Fall 2022
RespondEvaluateRememberDecodeReceive
--------------------------------------------------
Document 3:
EFFECTIVE Listening
 BC (HS-218)
SE Computer Science – Fall 2022
--------------------------------------------------
Document 4:
•Implications of the message
•Speaker intentions/ motives
Critical Listening
•Understand speaker’s feelings, needs and wants
•Listening with empathyEmpathic Listening
--------------------------------------------------


In [41]:
from langchain_core.prompts import ChatPromptTemplate

template = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use ten sentences maximum and keep the answer concise.

Question: {question}
Context: {context}
Answer:
"""

In [42]:
prompt=ChatPromptTemplate.from_template(template)

In [43]:
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf you don't know the answer, just say that you don't know.\nUse ten sentences maximum and keep the answer concise.\n\nQuestion: {question}\nContext: {context}\nAnswer:\n"), additional_kwargs={})])

In [45]:
from langchain_core.output_parsers import StrOutputParser

In [46]:
output_parser=StrOutputParser()

In [49]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Gemini LLM. Uses GOOGLE_API_KEY / GEMINI_API_KEY from the .env setup cell.
llm_model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.2,
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [50]:
from langchain_core.runnables import RunnablePassthrough


rag_chain = (
    {"context": retriever,  "question": RunnablePassthrough()}
    | prompt
    | llm_model
    | output_parser
)

In [51]:
rag_chain.invoke("What Makes an Effective Listener?")

"An effective listener prepares for the interaction, makes the listening meaningful, and uses the results effectively. They follow agreed-on rules, encourage participation, and actively engage while minimizing distractions. Such a listener can also close the communication effectively. Furthermore, an effective listener employs various listening modes. This includes content listening, where they focus purely on information without evaluation and remain neutral. They also engage in critical listening by understanding the logic, evaluating the strength of evidence, and discerning the implications of the message and speaker's intentions. Finally, an effective listener practices empathic listening to understand the speaker's feelings, needs, and wants with genuine empathy."

In [ ]:
import structlog

ModuleNotFoundError: No module named 'structlog'